Yes, when using `RetrievalQA` from LangChain, the chain primarily relies on the information retrieved by the **retriever** you provide (e.g., a vector store retriever like `VectorStoreRetriever`). 

### How `RetrievalQA` Works:
1. **Retrieval Step**: The retriever fetches relevant documents/chunks based on the input query.
2. **Generation Step**: The retrieved documents are passed to the language model (LLM), which synthesizes an answer using the provided context.

### Key Points:
- **Only Uses Retrieved Info**: By default, `RetrievalQA` does not use the LLM's internal knowledge (unless explicitly allowed or if the retriever fails to fetch relevant content). It grounds the answer strictly in the retrieved documents.
- **No Additional Knowledge**: If the retriever doesn't return relevant context, the LLM may struggle to generate a good answer unless it has inherent knowledge of the topic (which is typically suppressed in this setup).
- **Customization**: You can configure it to include the LLM's own knowledge (e.g., by tweaking `RetrievalQA.from_chain_type()` parameters or using `stuff`, `map_reduce`, etc.), but the default behavior is retrieval-first.

### Example:
```python
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

# Assume `retriever` is already defined (e.g., from a vector store)
qa = RetrievalQA.from_chain_type(
    llm=OpenAI(),
    chain_type="stuff",  # Other options: "map_reduce", "refine", etc.
    retriever=retriever,
    return_source_documents=True  # Optional: to see retrieved docs
)

result = qa("What is LangChain?")
print(result["answer"])  # Answer is based ONLY on retrieved documents.
```

### If You Want Hybrid Knowledge:
- Use `RetrievalQAWithSourcesChain` or a custom chain to blend retrieval + LLM knowledge.
- Adjust the prompt to explicitly allow the LLM to use its own knowledge if retrieval fails.

Let me know if you'd like details on configuring this behavior!

____________________________
Great question! 🎯 Here’s a step-by-step guide on how to **pull the Mistral model using Ollama** and **use it with LangChain**.

---

### 🧠 Step 1: Install & Run Ollama

If you haven’t already:

```bash
# Download and install Ollama
curl -fsSL https://ollama.com/install.sh | sh
```

Then, start Ollama:

```bash
ollama serve
```

---

### 🔽 Step 2: Pull the Mistral Model

Ollama supports several models, including **Mistral**.

```bash
ollama pull mistral
```

That’s it! Now Mistral is running locally via Ollama.

---

### 🧱 Step 3: Use Mistral in LangChain

LangChain has an `Ollama` wrapper you can use to connect to the local model.

#### Install dependencies:

```bash
pip install langchain langchain-community
```

#### Python Code Example:

```python
from langchain_community.llms import Ollama

# Load Mistral model from Ollama
llm = Ollama(model="mistral")

# Simple query
response = llm.invoke("Summarize the main themes in this survey response: 'I love the new design but it's still too slow.'")
print(response)
```

---

### 💡 Bonus: Use Mistral with LangChain RAG

If you want to use Mistral as part of a RAG pipeline (with embeddings + retriever + LLM), you can plug it in like this:

```python
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS  # or Chroma, etc.
from langchain.embeddings import HuggingFaceEmbeddings

# 1. Load your documents and create embeddings
# Assuming you've already split and embedded your docs

vectorstore = FAISS.load_local("faiss_index", embeddings=HuggingFaceEmbeddings())
retriever = vectorstore.as_retriever()

# 2. Load Mistral via Ollama
llm = Ollama(model="mistral")

# 3. Build the RAG chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# 4. Ask a question
query = "What are customers saying about checkout speed?"
answer = qa_chain.run(query)
print(answer)
```

---

### ✅ Summary

| Step | Action |
|------|--------|
| 1 | Install and run Ollama |
| 2 | `ollama pull mistral` |
| 3 | Use `Ollama(model="mistral")` in LangChain |
| 4 | Integrate into LangChain's QA or RAG pipeline |

---

Let me know if you want a full RAG pipeline example using your survey responses!

In [1]:
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM,OllamaEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA, RetrievalQAWithSourcesChain
from typing import List

from pydantic import BaseModel, Field
from typing import Optional,List
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [2]:
df = pd.read_csv("sentiment_topic_clustering.csv")

In [3]:
df.columns

Index(['Question', 'value', 'sentiment_labels', 'sentiment_score',
       'dbscan_label', 'cleaned_value'],
      dtype='object')

In [3]:
questions = df["Question"].unique()

In [5]:
questions

array(["Q6b - Qual: Why do you feel this way about Ausgrid's transition to electric vehicles?",
       'Q24: Please explain why you feel this way about driving a battery EV.\n(Open-ended)',
       'Q28: Do you have any additional comments or suggestions about Ausgrid transitioning to electric vehicles?'],
      dtype=object)

In [9]:
text_list = df.loc[df["Question"]==questions[0],"value"].dropna().tolist()

In [10]:
llm = OllamaLLM(model="mistral")

In [5]:
# Convert to Document objects
documents = [Document(page_content=text,metadata={"source": "text"}) for text in text_list]
embeddings = OllamaEmbeddings(model="mistral")
# Create vector store
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

# Now ready for RAG
retriever = vectorstore.as_retriever()

NameError: name 'text_list' is not defined

In [11]:
# Custom prompt that restricts answers to the context
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI assistant. Use ONLY the following context to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
)

In [32]:
# Create the RAG question-answering chain
qa_chain = RetrievalQA.from_chain_type(
    # The LLM that will generate answers (Mistral via Ollama in this case)
    llm=OllamaLLM(
        model="mistral",  # 7B parameter model good for balance of speed/quality
        temperature=0,  # Controls creativity (0 = factual, 1 = creative)
        top_p=0.9,       # Nucleus sampling parameter
    ),
    
    # The chain type determines how retrieved documents are processed:
    chain_type="stuff",  # Other options: "map_reduce", "refine", "map_rerank"
    
    # The retriever that fetches relevant documents from the vector store
    retriever=vectorstore.as_retriever(
        search_type="similarity",  # Default, other option: "mmr" (Maximal Marginal Relevance)
        search_kwargs={"k": 600}     # Number of documents to retrieve
    ),
    chain_type_kwargs={"prompt": custom_prompt},
    # Optional parameters:
    return_source_documents=True,  # Include source docs in output
    verbose=True                  # Print debugging info
)

### General question: Why do you feel this way about Ausgrid's transition to electric vehicles?

In [16]:
questions

array(["Q6b - Qual: Why do you feel this way about Ausgrid's transition to electric vehicles?",
       'Q24: Please explain why you feel this way about driving a battery EV.\n(Open-ended)',
       'Q28: Do you have any additional comments or suggestions about Ausgrid transitioning to electric vehicles?'],
      dtype=object)

In [17]:
questions[0][12:]

"Why do you feel this way about Ausgrid's transition to electric vehicles?"

### Prompt 1 Q6b

In [18]:
prompt = """
please provide me top 30 unigrams, bigrams and trigrams
"""

In [33]:
result = qa_chain.invoke("please provide me top 30 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Unigrams: electric, vehicles, environment, charging, Sydney, infrastructure, operational, downtime, coal, diesel, batteries, lithium, slave, emissions, green, 24/7, non-productive, time, accessibility, units, costs, hurdles, distribution, technicians, climate, change, practical, future, lead, workforce

Bigrams: electric vehicles, environment charging, Sydney infrastructure, operational downtime, coal life, batteries lithium, slave labor, emissions green, 24/7 operation, non-productive time, accessibility issues, distribution uneven, technicians shortage, cleaner diesel, battery life, cradle to grave, some work suitable, future lead way, operational challenges

Trigrams: electric vehicles environment, charging infrastructure Sydney, operational downtime costs, coal life expectancy, batteries lithium emissions, slave labor greenhouse, 24/7 operation non-productive, time accessibility issues, distribution uneven Sydney, technician

### Prompt 2 Q6b

In [34]:
result = qa_chain.invoke("please provide the counts for each of the top 30 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 30 Unigrams:
1. electric (24)
2. vehicles (23)
3. environment (22)
4. charging (21)
5. ev (20)
6. australia (19)
7. sydney (18)
8. infrastructure (17)
9. operation (16)
10. time (16)
11. benefits (15)
12. challenges (14)
13. costs (14)
14. downtime (13)
15. accessibility (13)
16. units (13)
17. productivity (12)
18. emissions (12)
19. co2 (12)
20. rare earth (11)
21. batteries (11)
22. 24/7 (11)
23. operational (11)
24. green (10)
25. renewable (10)
26. sustainability (9)
27. cleaner (9)
28. climate change (9)
29. practical (9)
30. financial (9)

Top 30 Bigrams:
1. electric vehicles (24)
2. environment benefits (22)
3. charging infrastructure (21)
4. operational challenges (18)
5. time efficiency (16)
6. co2 emissions (15)
7. rare earth metals (14)
8. battery life (13)
9. sydney australia (13)
10. charging stations (12)
11. ev charging (12)
12. 24/7 operation (11)
13. increased costs (11)
14. non-productive time (11)
15. acc

### Prompt 3 Q6b

In [36]:
result = qa_chain.invoke("please provide the sentiment for each of the top 10 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Unigram Sentiment Analysis:
1. ev: Positive (67%)
2. environment: Positive (83%)
3. charging: Neutral (50%)
4. infrastructure: Negative (40%)
5. operational: Negative (50%)
6. Sydney: Neutral (50%)
7. time: Neutral (50%)
8. cost: Negative (60%)
9. downtime: Negative (80%)
10. challenges: Negative (60%)

Bigram Sentiment Analysis:
1. ev charging: Neutral (50%)
2. environment benefits: Positive (83%)
3. operational delays: Negative (70%)
4. Sydney infrastructure: Negative (60%)
5. time efficiency: Neutral (50%)
6. cost downtime: Negative (70%)
7. charging locations: Negative (60%)
8. accessibility charging: Negative (60%)
9. increased costs: Negative (80%)
10. operational complexities: Negative (70%)

Trigram Sentiment Analysis:
1. shift to electric vehicles: Positive (53%)
2. current infrastructure and: Neutral (50%)
3. environmental benefits but: Neutral (50%)
4. operational challenges in: Negative (67%)
5. limited number of fas

### Prompt 4 Q6b

In [38]:
result = qa_chain.invoke("please provide the top 10 reasons on the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Better for the environment (14 responses)
2. Fuel efficiency (9 responses)
3. Aligns with strategic vision (7 responses)
4. Reduces emissions & fuel bills (6 responses)
5. Electricity company (5 responses)
6. Climate change (5 responses)
7. Convenient for short distance travel (4 responses)
8. No petrol stations or fuel cards needed (4 responses)
9. New technology that makes sense (3 responses)
10. On track with the future (3 responses)


### Prompt 5 Q6b

In [39]:
result = qa_chain.invoke("please provide the top 10 barriers/challenges and opportunities to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 10 Barriers/Challenges:

1. Lack of charging infrastructure: Many employees are concerned about the availability of charging stations, especially in remote locations or buildings without chargers.
2. Range anxiety: Employees who travel long distances daily may worry about the need to charge their vehicles every day or every second day.
3. Battery life and longevity: There is a concern that electric vehicle batteries will need replacing frequently, which could be costly and environmentally unfriendly.
4. Productivity time loss: Charging an electric vehicle can take longer than refueling a petrol or diesel vehicle, potentially affecting productivity.
5. Financial/economic concerns: Some employees believe the additional cost to purchase and install charging infrastructure for an electric vehicle is not justified.
6. Environmental impact of batteries: There are concerns about the environmental impact of battery production, parti

### Prompt 6 Q6b

In [40]:
result = qa_chain.invoke("please provide the top 10 interventions to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Provide clear and concise information about the benefits of electric vehicles (EVs) for both the environment and cost savings in the long run.
2. Address concerns about battery lifecycle, recycling, and the environmental impact of EV production.
3. Demonstrate that there are suitable charging infrastructure solutions available, especially away from home and workplaces.
4. Showcase real-world examples of successful EV implementation for various use cases, including tradesmen and long-distance travel.
5. Offer financial incentives or subsidies to offset the initial cost of purchasing an electric vehicle.
6. Address safety concerns related to EVs by providing accurate information about incidents and addressing misconceptions.
7. Collaborate with industry partners, government agencies, and other stakeholders to create a comprehensive plan for the transition to EVs.
8. Provide training and support for employees who will be using el

### Prompt 7 Q6b

In [41]:
result = qa_chain.invoke("please provide the top 5 items for Psychological Capability and Physical Capability in COM-B behavioural change model to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Psychological Capability:
   - Provide clear, concise, and compelling information about the environmental benefits of electric vehicles (EVs) and how they align with Ausgrid's mission and values.
   - Address concerns about the convenience and practicality of EVs by providing solutions for charging infrastructure, range anxiety, and battery lifecycle management.
   - Encourage employees to share their personal experiences with EVs, both positive and negative, to foster a sense of community and collaboration in finding solutions.
   - Offer training and support for employees who are new to EVs, such as workshops on how to charge, maintain, and troubleshoot the vehicles.
   - Highlight the financial benefits of EVs, such as lower fuel costs and reduced maintenance needs, to demonstrate the practicality and long-term savings associated with the transition.

2. Physical Capability:
   - Ensure that the EV models chosen for Ausgrid

### Prompt 8 Q6b

In [45]:
result = qa_chain.invoke("please provide the top 5 items for each Physical Opportunity, Social Opportunity in COM-B behavioural change model to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Physical Opportunity:
   - Provide practical information about the range, charging time, and cost of electric vehicles for different types of work (trades vs office use).
   - Demonstrate the longevity and durability of electric vehicles, addressing concerns about battery lifecycle and replacement costs.
   - Showcase the fuel efficiency and environmental benefits of electric vehicles compared to traditional vehicles.
   - Offer solutions for charging infrastructure away from depots, particularly for long-distance travel.
   - Provide data on the safety records of electric vehicles to address concerns about explosions and accidents.

2. Social Opportunity:
   - Facilitate open discussions and Q&A sessions with experts to address misconceptions and concerns about electric vehicles.
   - Share success stories and case studies of other companies that have successfully transitioned to electric fleets.
   - Encourage peer-to-peer l

### Prompt 9 Q6b

In [43]:
result = qa_chain.invoke("please provide the top 5 items for  Reflective Motivation and Automatic Motivation in COM-B behavioural change model  to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Reflective Motivation:
   - Understanding the environmental benefits of electric vehicles (EVs) and how they align with Ausgrid's commitment to sustainability.
   - Acknowledging the economic advantages of EVs, such as lower operating costs and potential tax incentives.
   - Recognizing the positive impact EVs can have on public perception and Ausgrid's reputation as a forward-thinking organization.
   - Emphasizing the role models set by other companies and governments that have successfully transitioned to EV fleets.
   - Encouraging open dialogue about the challenges and solutions associated with EV adoption, fostering a sense of shared responsibility and commitment.

2. Automatic Motivation:
   - Making EVs easily accessible and convenient for employees by providing charging infrastructure at workplaces and homes.
   - Offering incentives such as subsidies or rebates to encourage the purchase of EVs.
   - Providing trainin

### Prompt 10 Q6b

In [44]:
result = qa_chain.invoke("please provide the top 5 items for Behavioural Changes component in COM-B behavioural change model to improve the level of agreement on choosing Ausgrid’s transition to electric vehicles")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Education and Information: Provide comprehensive, accessible, and easy-to-understand information about the benefits of electric vehicles (EVs), their environmental impact, battery lifecycle, charging infrastructure, and cost-effectiveness compared to traditional vehicles. This could include case studies, demonstrations, and interactive sessions.

2. Social Influence: Encourage peer-to-peer learning and sharing experiences among employees who have already transitioned to EVs or are considering it. This can help build trust and confidence in the technology and its practicality for work purposes.

3. Reinforcement: Offer incentives, such as subsidies on the purchase of EVs, free charging at work, or priority parking spaces for EV owners. This can help offset any initial financial concerns and make the transition more appealing.

4. Goal Setting: Establish clear goals and timelines for the transition to EVs within the organization

# General question: Q24: Please explain why you feel this way about driving a battery EV.\n(Open-ended)

In [12]:
text_list = df.loc[df["Question"]==questions[1],"value"].dropna().tolist()

In [13]:
# Convert to Document objects
documents = [Document(page_content=text) for text in text_list]
embeddings = OllamaEmbeddings(model="mistral")
# Create vector store
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

# Now ready for RAG
retriever = vectorstore.as_retriever()

In [14]:
# Create the RAG question-answering chain
qa_chain = RetrievalQA.from_chain_type(
    # The LLM that will generate answers (Mistral via Ollama in this case)
    llm=OllamaLLM(
        model="mistral",  # 7B parameter model good for balance of speed/quality
        temperature=0.3,  # Controls creativity (0 = factual, 1 = creative)
        top_p=0.9,       # Nucleus sampling parameter
    ),
    
    # The chain type determines how retrieved documents are processed:
    chain_type="stuff",  # Other options: "map_reduce", "refine", "map_rerank"
    
    # The retriever that fetches relevant documents from the vector store
    retriever=vectorstore.as_retriever(
        search_type="similarity",  # Default, other option: "mmr" (Maximal Marginal Relevance)
        search_kwargs={"k": 600}     # Number of documents to retrieve
    ),
    chain_type_kwargs={"prompt": custom_prompt},
    # Optional parameters:
    return_source_documents=True,  # Include source docs in output
    verbose=True                  # Print debugging info
)

### Prompt 1 Q24

In [15]:
result = qa_chain.invoke("please provide me top 30 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Unigrams: car, drive, electric, vehicle, environment, fuel, work, company, km, charge, range, battery, diesel, city, day, task, gear, future, technology, ideology, productivity, unscheduled, reactive, unfamiliar, experience, quiet, cheaper, refueling, emissions, safety

Bigrams: drive car, electric vehicle, work company, fuel costs, environment friendly, km km's, charge battery, range limit, battery life, diesel diesel vehicles, city city commute, day office based, task remote, gear equipment, future driving, technology adoption, ideology practicality, productivity ev, reactive unscheduled

Trigrams: drive car experience, electric vehicle environment, work company fuel costs, km km's travel, charge battery life, range limit capacity, diesel vehicles emissions, city commute traffic, day office based tasks, task remote locations, gear equipment carry, future driving technology, technology adoption benefits, ideology practicality c

### Prompt 2 Q24

In [16]:
result = qa_chain.invoke("please provide the counts for each of the top 30 unigrams, bigrams and trigrams excluding stopwords")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 30 Unigrams (excluding stopwords):
1. ev - 96
2. vehicle - 47
3. driving - 38
4. company - 35
5. work - 34
6. truck - 33
7. charge - 30
8. change - 27
9. role - 26
10. electric - 25
11. future - 24
12. experience - 23
13. time - 22
14. day - 22
15. life - 21
16. use - 20
17. road - 20
18. fuel - 19
19. city - 19
20. office - 18
21. km - 18
22. unplanned - 18
23. storms - 17
24. faults - 17
25. options - 16
26. technology - 16
27. safety - 16
28. environment - 16
29. green - 15
30. unigrams - 15

Top 30 Bigrams (excluding stopwords):
1. ev vehicle - 9
2. driving experience - 7
3. company role - 6
4. work day - 6
5. truck charge - 5
6. change future - 5
7. electric vehicle - 5
8. time life - 4
9. unplanned day - 4
10. storms faults - 4
11. fuel road - 4
12. city office - 3
13. km week - 3
14. other options - 3
15. technology change - 3
16. safety features - 3
17. environment friendly - 3
18. green environmental - 3
19. unplann

### Prompt 3 Q24

In [22]:
result = qa_chain.invoke("please provide the sentiment for each of the top 10 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Unigram Sentiments:
1. "ev" - Mixed (Some are open to it, while others are not fans)
2. "company" - Positive (Most believe it's a good step for the company)
3. "driving" - Mixed (Some prefer driving an EV, while others do not)
4. "work" - Mixed (Views on using an EV for work are divided)
5. "car" - Mixed (Some want an EV if it suits their requirements, while others do not)
6. "future" - Positive (Many believe EVs represent the future of driving)
7. "fuel" - Negative (Some struggle with refueling and are open to cheaper charging costs)
8. "charging" - Mixed (Views on charging are divided, some find it inconvenient)
9. "range" - Negative (Concerns about EV range, especially for long distances)
10. "electric" - Mixed (Some see the environmental benefits, while others are not fans)

Bigram Sentiments:
1. "ev culture" - Negative (Some are not fans of the 'woke' EV culture)
2. "embrace change" - Positive (Many are open to adapting to 

### Prompt 4 Q24

In [23]:
result = qa_chain.invoke("please provide the top 10 reasons on feeling for the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Saving the company money in fuel costs and helping the environment.
2. Quiet, nicer vehicles generally.
3. A new experience and embracing change and technology.
4. Adapting to suit the future and electrify everything.
5. Modern and full of safety features compared to an old vehicle.
6. Environmentally friendlier, more safety options are available.
7. Good for green environmental purposes.
8. Happy to adopt new technology and move with the times.
9. Passionate about supporting the energy transition to net zero.
10. Feeling futuristic and wanting to have an experience of EVs.


### Prompt 5 Q24

In [18]:
result = qa_chain.invoke("please provide the top 10 barriers/challenges and opportunities to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 10 Barriers/Challenges:

1. Limited charging infrastructure in remote areas and off-road locations.
2. Uncertainty about battery life and range during long distance travel.
3. Fear of getting stranded due to a dead battery or lack of charging stations.
4. Concerns about the accuracy of battery life indicators compared to actual kilometers traveled.
5. Preference for hybrid vehicles over fully electric ones.
6. Lack of confidence in the durability and functional characteristics of EVs, especially for heavy-duty work.
7. Unfamiliarity with the technology and lack of personal experience driving an EV.
8. Limited charging opportunities at designated parking lots or during long shifts.
9. Concerns about the cost and time required to recharge an EV compared to refueling a traditional vehicle.
10. Skepticism towards the effectiveness of EVs in addressing climate change.

Top 10 Opportunities:

1. Improving charging infrastructure, 

### Prompt 6 Q24

In [19]:
result = qa_chain.invoke("please provide the top 10 interventions to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Provide accurate and reliable range estimates based on real-world conditions, including load capacity and terrain.
2. Improve charging infrastructure, particularly in remote areas where long-distance travel is required.
3. Offer hybrid vehicles as an option for those who require take-home use or off-road capabilities.
4. Ensure the EV has adequate storage space for tools, equipment, and rescue gear commonly used in the role.
5. Address concerns about battery life and reliability through warranties, maintenance plans, and clear communication about expected lifespan and performance.
6. Offer online training modules to build confidence in operating and maintaining the EV.
7. Highlight the safety features and modern technology of EVs compared to older, outdated vehicles.
8. Emphasize the environmental benefits of driving an EV, including reduced emissions and energy savings.
9. Address concerns about the cost-effectiveness of EVs 

### Prompt 7 Q24

In [24]:
result = qa_chain.invoke("please provide the top 5 items for Psychological Capability and Physical Capability in COM-B behavioural change model to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Psychological Capability:
   - Provide comprehensive training and education about the operation, maintenance, and safety features of the battery electric vehicle (BEV). This includes understanding charging times and locations, range estimation, and any unique aspects of driving a BEV.
   - Address concerns about reliability and durability by providing data and evidence that demonstrates the long-term performance of BEVs in demanding work conditions.
   - Foster a positive attitude towards BEVs by emphasizing their benefits, such as quieter operation, reduced emissions, and improved safety features.
   - Encourage open dialogue and feedback to address any concerns or issues that may arise during the transition to driving a BEV.
   - Offer incentives and rewards for adopting and successfully using a BEV for work purposes.

2. Physical Capability:
   - Ensure that the BEV is equipped with adequate storage solutions for tools, equ

### Prompt 8 Q24

In [25]:
result = qa_chain.invoke("please provide the top 5 items for each Physical Opportunity, Social Opportunity in COM-B behavioural change model to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Physical Opportunity:
   - Providing charging infrastructure at work and along common travel routes.
   - Offering electric vehicles with sufficient range for daily work requirements.
   - Ensuring the electric vehicle is equipped with necessary tools and gear for the job.
   - Demonstrating the ease of use, efficiency, and performance of electric vehicles compared to traditional ones.
   - Addressing concerns about battery life and charging times by providing accurate information and solutions.

2. Social Opportunity:
   - Encouraging a supportive work culture that values environmental responsibility and innovation.
   - Providing opportunities for employees to share experiences, tips, and best practices related to electric vehicles.
   - Offering training programs to help employees become more familiar with the technology and feel confident driving an electric vehicle.
   - Creating a sense of community among those who drive

### Prompt 9 Q24

In [26]:
result = qa_chain.invoke("please provide the top 5 items for Reflective Motivation and Automatic Motivation in COM-B behavioural change model to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Reflective Motivation:
   - Environmental Concerns: Understanding the impact of traditional vehicles on the environment and the benefits of using an electric vehicle (EV) in terms of reducing carbon footprint, air pollution, and energy conservation.
   - Safety Features: Awareness of the advanced safety features available in EVs, such as autonomous emergency braking, lane-keeping assist, and adaptive cruise control, which can enhance road safety.
   - Cost Savings: Recognizing the long-term cost savings associated with owning and operating an EV, including lower fuel costs, reduced maintenance expenses, and potential tax incentives.
   - Personal Values: Aligning personal values with the adoption of green technology and supporting a more sustainable future for the company and society as a whole.
   - Improved Driving Experience: Appreciating the quieter, smoother ride and overall better driving experience offered by EVs compar

### Prompt 10 Q24

In [27]:
result = qa_chain.invoke("please provide the top 5 items for Behavioural Changes in COM-B behavioural change model to improve the level of willingness to choose driving a battery EV for work if one were made available")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Increase knowledge and understanding about battery electric vehicles (BEVs): Provide comprehensive information about BEVs, their benefits, charging infrastructure, and how they can meet specific job requirements. This could be through online training modules, demonstrations, or hands-on experiences.

2. Address concerns about range anxiety: Ensure that employees understand the realistic range capabilities of available BEV models and provide strategies for managing range during long trips, such as route planning with charging stations along the way or having backup gasoline vehicles available for emergencies.

3. Emphasize the environmental benefits of BEVs: Highlight the positive impact of BEVs on reducing greenhouse gas emissions, improving air quality, and contributing to a more sustainable future. This can help create a sense of responsibility and motivation among employees to adopt more eco-friendly practices.

4. Provide 

# General question: Q28: Do you have any additional comments or suggestions about Ausgrid transitioning to electric vehicles?'

In [28]:
text_list = df.loc[df["Question"]==questions[2],"value"].dropna().tolist()

In [29]:
# Convert to Document objects
documents = [Document(page_content=text) for text in text_list]
embeddings = OllamaEmbeddings(model="mistral")
# Create vector store
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

# Now ready for RAG
retriever = vectorstore.as_retriever()

In [30]:
# Create the RAG question-answering chain
qa_chain = RetrievalQA.from_chain_type(
    # The LLM that will generate answers (Mistral via Ollama in this case)
    llm=OllamaLLM(
        model="mistral",  # 7B parameter model good for balance of speed/quality
        temperature=0.3,  # Controls creativity (0 = factual, 1 = creative)
        top_p=0.9,       # Nucleus sampling parameter
    ),
    
    # The chain type determines how retrieved documents are processed:
    chain_type="stuff",  # Other options: "map_reduce", "refine", "map_rerank"
    
    # The retriever that fetches relevant documents from the vector store
    retriever=vectorstore.as_retriever(
        search_type="similarity",  # Default, other option: "mmr" (Maximal Marginal Relevance)
        search_kwargs={"k": 600}     # Number of documents to retrieve
    ),
    chain_type_kwargs={"prompt": custom_prompt},
    # Optional parameters:
    return_source_documents=True,  # Include source docs in output
    verbose=True                  # Print debugging info
)

### Prompt 1 Q28

In [31]:
result = qa_chain.invoke("please provide me top 30 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Unigrams: electric, vehicle, charging, depot, staff, business, company, range, petrol, counterparts, maintenance, parts, complex, operation, different, beast, simpler, operate, select, match, vast, distance, juice-ups, skilled, technicians, parts, access, battery, solar, discharge, slow, fast, bidirectional, network, peak, investment, sound, wired, good

   Bigrams: electric vehicle, charging depot, staff business, company range, petrol counterparts, maintenance parts, complex operation, different beast, simpler operate, operate counterparts, select vehicles, match business, operational range, juice-ups distance, skilled technicians, right parts, access technicians, battery solar, discharge capacity, slow chargers, fast chargers, bidirectional charging, large numbers, average self consumption

   Trigrams: electric vehicle charging, depot staff business, company range petrol, counterparts maintenance parts, complex operation dif

### Prompt 2 Q28

In [33]:
result = qa_chain.invoke("please provide the counts for each of the top 30 unigrams, bigrams and trigrams excluding stopwords, with count")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 30 Unigrams (excluding stopwords):
1. yes: 24
2. no: 22
3. electric: 17
4. ev: 16
5. good: 15
6. time: 15
7. car: 14
8. charger: 13
9. vehicle: 13
10. drive: 12
11. day: 12
12. company: 11
13. miles: 11
14. hour: 11
15. battery: 10
16. km: 9
17. staff: 8
18. depot: 8
19. work: 8
20. system: 8
21. fuel: 7
22. fire: 7
23. incident: 7
24. storm: 7
25. outage: 6
26. capacity: 6
27. inconvenience: 6
28. cost: 6
29. experience: 6
30. range: 6

Top 30 Bigrams (excluding stopwords):
1. yes electric: 14
2. no electric: 11
3. electric car: 10
4. electric vehicle: 8
5. good time: 7
6. time drive: 6
7. drive day: 6
8. company staff: 5
9. miles hour: 5
10. hour miles: 5
11. charger car: 4
12. charger depot: 4
13. depot hour: 4
14. vehicle battery: 4
15. battery system: 4
16. fuel electric: 4
17. incident fire: 3
18. storm outage: 3
19. outage days: 3
20. cost time: 3
21. km miles: 3
22. staff company: 3
23. drive hours: 3
24. hour capaci

### Prompt 3 Q28

In [34]:
result = qa_chain.invoke("please provide the sentiment for each of the top 10 unigrams, bigrams and trigrams")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Unigrams:
- Electric (Positive)
- Vehicle (Neutral)
- Charge (Positive)
- Staff (Neutral)
- Depot (Neutral)
- Work (Neutral)
- EV (Positive)
- Car (Neutral)
- Time (Neutral)
- Cost (Negative)

2. Bigrams:
- Electric vehicle (Positive)
- Charge staff (Positive)
- Depot charging (Neutral)
- Work depot (Neutral)
- EV market (Neutral)
- Cost investment (Negative)
- Staff parking (Neutral)
- Overnight charging (Neutral)
- Fast chargers (Neutral)
- Peak times (Neutral)

3. Trigrams:
- Electric vehicle market (Neutral)
- Charge staff parking (Positive)
- Depot charging station (Neutral)
- Work depot location (Neutral)
- EV charging infrastructure (Neutral)
- Cost investment strategy (Negative)
- Staff parking facilities (Neutral)
- Overnight charging system (Neutral)
- Fast charger network (Neutral)
- Peak times management (Neutral)


### Prompt 4 Q28

### Prompt 5 Q28

In [37]:
result = qa_chain.invoke("please provide the top 10 barriers/challenges and opportunities for EV fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
 Top 10 Barriers/Challenges for EV Fleets:

1. Limited charging infrastructure, especially at destinations and remote locations.
2. Higher upfront costs compared to traditional Internal Combustion Engine (ICE) vehicles.
3. Longer refueling times compared to ICE vehicles.
4. Lower driving range on a single charge, which may not be suitable for long-distance travel or heavy usage.
5. Limited model options and availability of EVs that meet specific fleet needs.
6. Potential negative impact on vehicle performance in extreme temperatures.
7. Higher maintenance costs due to fewer service providers familiar with EVs.
8. Battery degradation over time, which can affect the lifespan and resale value of EVs.
9. Lack of standardization in charging protocols and compatibility issues between different EV models.
10. Public perception and skepticism about the reliability and practicality of EVs.

Top 10 Opportunities for EV Fleets:

1. Reduced 

### Prompt 6 Q28

In [38]:
result = qa_chain.invoke("please provide the top 10 interventions to foster EV Fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Provide sufficient charging infrastructure at depots, with designated spaces for fleet vehicles and clear rules about their use.
2. Offer employee chargers that can slow charge all day without contention, as well as faster chargers for those who need them.
3. Encourage the use of EVs in the general pool car fleet, and increase the number of these cars to meet demand.
4. Allow employees to take home EVs on weekends to increase familiarity and comfort with the vehicles.
5. Consider purchasing cheaper EV models, such as MG or BYD, to reduce costs.
6. Provide training in charger etiquette to prevent contention and ensure efficient use of charging resources.
7. Offer a variety of EV models for employees to choose from, so that the best one can be found for the company's needs.
8. Trial heavy vehicles in EV form to see their potential benefits for night work and other applications.
9. Set guidelines for when and how pool vehicles sh

### Prompt 7 Q28

In [39]:
result = qa_chain.invoke("please provide the top 5 items for Psychological Capability and Physical Capability in COM-B behavioural change model to foster EV Fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Psychological Capability:
   - Education and Training: Provide comprehensive training on the operation, maintenance, and safety of electric vehicles (EVs). This includes understanding charging etiquette, range anxiety management, and the benefits of EVs for both the individual and the organization.
   - Self-efficacy: Encourage a sense of confidence in the ability to effectively use and maintain EVs by providing clear instructions, troubleshooting guides, and easy access to technical support.
   - Attitude: Foster a positive attitude towards EVs by emphasizing their environmental benefits, cost savings over time, and potential for job security in a growing industry.

2. Physical Capability:
   - Accessibility: Ensure that charging infrastructure is easily accessible at all depots, with a mix of slow and fast chargers to accommodate different needs. This includes providing off-street parking for those without it at home.
   - M

### Prompt 8 Q28

In [40]:
result = qa_chain.invoke("please provide the top 5 items for each Physical Opportunity, Social Opportunity in COM-B behavioural change model to foster EV Fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Physical Opportunity:
   - Provide sufficient charging infrastructure at workplaces and along travel routes.
   - Offer a variety of EV models suitable for different job requirements.
   - Install chargers at employees' homes to support home charging.
   - Ensure that EV pool cars are available and easily accessible.
   - Regularly maintain and service EVs to ensure they are reliable and efficient.

2. Social Opportunity:
   - Implement training programs on charger etiquette to prevent contention and promote fair usage.
   - Encourage carpooling and ride-sharing among employees.
   - Establish a system for sharing charging spots, especially for personal EVs.
   - Foster a culture that values environmental sustainability and encourages the adoption of EVs.
   - Create opportunities for employees to share experiences and best practices about EV usage.


### Prompt 9 Q28

In [41]:
result = qa_chain.invoke("please provide the top 5 items for Reflective Motivation and Automatic Motivation in COM-B behavioural change model to foster EV Fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Reflective Motivation:
   - Understanding the environmental benefits of electric vehicles (EVs) and how they align with Ausgrid's commitment to sustainability.
   - Recognizing the cost savings in the long run due to lower fuel and maintenance costs associated with EVs.
   - Acknowledging the potential for improved employee satisfaction by offering modern, efficient, and eco-friendly vehicles as part of their work experience.
   - Realizing the positive impact on public perception as Ausgrid adopts more environmentally friendly practices.
   - Being aware of the potential for reduced downtime due to fewer mechanical issues with EVs compared to traditional Internal Combustion Engine (ICE) vehicles.

2. Automatic Motivation:
   - Making charging infrastructure easily accessible at workplaces and homes, making it convenient for employees to charge their EVs.
   - Providing training on charger etiquette to minimize contention and 

### Prompt 10 Q28

In [42]:
result = qa_chain.invoke("please provide the top 5 items for Behavioural Changes in COM-B behavioural change model to foster EV Fleets")
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
1. Capability: Provide training and resources to employees on electric vehicle (EV) operation, charging etiquette, and maintenance. This includes understanding how to charge an EV efficiently and safely, as well as troubleshooting common issues.

2. Opportunity: Ensure that there are sufficient charging stations at workplaces and destinations, both for fleet vehicles and personal EVs. This will help reduce contention and make it more convenient for employees to use EVs.

3. Motivation: Highlight the benefits of using EVs, such as cost savings, environmental impact reduction, and improved health due to reduced air pollution. Offering incentives like subsidies or preferential parking can also motivate employees to adopt EVs.

4. Reinforcement: Implement a system for sharing information about available charging stations, charging status, and best practices among employees. This could be through a dedicated app, email updates, or a s